# DQN 期货配对交易研究

本 Notebook 负责研究问题、实验编排和结果解释。可复用实现位于 `src/rl_pairtrading`，参数由 Pydantic 校验。

默认使用 `configs/smoke.toml` 完成快速流程验证；正式实验请将 `CONFIG_PATH` 指向 `configs/default.toml` 或自建配置。快速配置的收益没有研究结论意义。

运行前在项目根目录执行 `uv sync --extra notebook --group dev`，再选择 `.venv` 的 Python 内核。参考 [README](../README.md)、[架构](../docs/architecture.md)、[研究口径](../docs/research.md)。

In [24]:
import logging
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from rl_pairtrading.agents.dqn import DQNAgent
from rl_pairtrading.artifacts import create_run_directory
from rl_pairtrading.backtesting.engine import BacktestEngine
from rl_pairtrading.backtesting.metrics import calculate_metrics
from rl_pairtrading.config import load_config
from rl_pairtrading.data.repository import LocalDataRepository, align_pair
from rl_pairtrading.domain import Action
from rl_pairtrading.environment.pair_trading import PairTradingEnv
from rl_pairtrading.training.trainer import Trainer, evaluate
from rl_pairtrading.training.walk_forward import run_walk_forward
from rl_pairtrading.training.windows import walk_forward_windows

# 从项目根目录或 notebook/ 启动时，均向上定位项目根目录。
ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "default.toml").exists()
)
CONFIG_PATH = ROOT / os.environ.get("RPT_CONFIG", "configs/smoke.toml")
config = load_config(CONFIG_PATH, project_root=ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
run_dir = create_run_directory(config)
print(f"配置：{CONFIG_PATH.name} | 结果目录：{run_dir}")
display(pd.Series(config.training.model_dump(), name="training"))

配置：smoke.toml | 结果目录：/Users/hxy/Documents/Project/RL_PairTrading/artifacts/20261106T071135_0aa9a1b6


seed                      41.0
episodes                   2.0
target_update_episodes     1.0
patience                   2.0
min_improvement            0.0
Name: training, dtype: float64

## 1. 研究问题与数据

问题：以无截距 OLS 估计的价差，能否通过 DQN 学到扣除交易成本后的持仓策略？

每个品种保留自身交易日；当前配对按共同日期对齐。不覆盖原始 pickle，不填造价格。全品种实验通过 `research.run_walk_forward` 控制。

In [ ]:
repository = LocalDataRepository(config)
data = repository.load_all()
contract_prices = repository.contract_prices()
first_symbol, second_symbol = config.research.primary_pair
first, second = align_pair(data[first_symbol], data[second_symbol])
display(
    pd.DataFrame(
        [
            {
                "symbol": symbol,
                "rows": len(frame),
                "start": frame.index.min(),
                "end": frame.index.max(),
            }
            for symbol, frame in data.items()
        ]
    ).set_index("symbol")
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(first.index, first.open, label=first_symbol)
axes[0].plot(second.index, second.open, label=second_symbol)
axes[0].set_ylabel("Open price")
axes[0].legend()
axes[1].plot(first.index, first.open - second.open, label="Raw spread (beta = 1)")
axes[1].set_ylabel("Raw spread")
axes[1].legend()
fig.tight_layout()
plt.show()

## 2. 时间切分

先展示一个窗口，检验训练、验证、测试流程。训练只更新参数，验证负责早停和选模，测试仅在选模完成后使用。

验证和测试允许携带此前 `lookback` 天历史作为特征预热；各分区末日仅用于观察结果，不在该日生成动作。测试切片多取一个结果日，因此不会遗漏每个窗口最后一个信号。

In [ ]:
windows = list(walk_forward_windows(len(first), config.environment.lookback, config.walk_forward))
if not windows:
    raise ValueError("共同交易日不足，请缩短配置中的窗口或增加历史数据")
window = windows[0]
parts = {"train": window.train, "validation": window.validation, "test": window.test}
display(
    pd.DataFrame(
        [
            {
                "partition": name,
                "warmup_start": first.index[part.start],
                "first_action": first.index[part.start + config.environment.lookback],
                "last_action": first.index[part.stop - 2],
                "last_observation": first.index[part.stop - 1],
            }
            for name, part in parts.items()
        ]
    ).set_index("partition")
)
environments = {
    name: PairTradingEnv(first.iloc[part], second.iloc[part], config.environment, contract_prices)
    for name, part in parts.items()
}

## 3. 训练与验证

网络结构、设备、探索率、经验回放和早停均由配置控制。自动设备选择顺序为 CUDA → MPS → CPU；快速验证配置固定使用 CPU。保存的是最佳验证权重，供推理或迁移学习使用，并非完整断点恢复。

In [ ]:
agent = DQNAgent(config.environment.state_size, config.agent, config.network, config.training.seed)
training = Trainer(config.training).fit(
    agent, environments["train"], environments["validation"], checkpoint=run_dir / "best_weights.pt"
)
training.history.to_csv(run_dir / "training_history.csv")
print(f"最佳回合：{training.best_episode} | 验证奖励：{training.best_validation_reward:.4f}")
training.history[["train_reward", "validation_reward"]].plot(figsize=(10, 4))
plt.ylabel("Spread reward")
plt.show()

## 4. 独立测试与交易信号

`LONG=0` 表示多第一腿、空第二腿；`SHORT=1` 相反；`CLOSE=2` 表示空仓。信号和对冲系数分列保存，不对整张表执行数值替换。

In [ ]:
evaluation = evaluate(environments["test"], agent)
signals = evaluation.signals.reset_index().assign(first=first_symbol, second=second_symbol)
signals.to_csv(run_dir / "signals.csv", index=False)
print(f"测试价差奖励：{evaluation.reward:.4f}")
display(signals.head())
fig, ax = plt.subplots(figsize=(12, 3))
for action in Action:
    chosen = signals[signals.action == int(action)]
    ax.scatter(chosen.date, chosen.action, label=action.name, s=12)
ax.set_yticks([int(action) for action in Action], [action.name for action in Action])
ax.legend()
plt.show()

## 5. 账户回测与解释

账户逐日结算，检查保证金，处理平仓、反向和主力换月；默认在最后一个信号日的结算价平仓。训练奖励是开盘到下一开盘的价差单位，账户曲线则包含合约乘数、数量和逐日结算，二者不是同一个指标。

默认合约乘数为 1，并允许小数头寸，延续旧版研究假设。收益展示用于检查实现；真实合约规格、整手约束、滑点和执行时点需另行研究。

In [ ]:
backtest = BacktestEngine(config, data, contract_prices).run(signals)
metrics = calculate_metrics(backtest.returns, config.backtest)
metrics.to_csv(run_dir / "metrics.csv")
backtest.trades.to_csv(run_dir / "trades.csv", index=False)
pd.concat([backtest.equity, backtest.cash, backtest.margin, backtest.returns], axis=1).to_csv(
    run_dir / "account.csv"
)
display(metrics.to_frame("value"))
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
backtest.equity.plot(ax=axes[0], title="Account equity")
(backtest.equity / backtest.equity.cummax().clip(lower=config.backtest.initial_cash) - 1).plot(
    ax=axes[1], title="Drawdown"
)
fig.tight_layout()
fig.savefig(run_dir / "account.png", dpi=150)
plt.show()

## 6. 多品种滚动研究（可选）

启用 `research.run_walk_forward = true` 后运行。每对品种、每个窗口创建独立模型，仅用训练段做协整筛选。未通过筛选的窗口输出空仓信号，避免旧持仓跨窗口残留。

这里不复用上面的预训练权重，避免将较晚时间训练的知识带到更早窗口。建议先用快速配置和少量品种验证，再扩大窗口和组合数量。

In [ ]:
if config.research.run_walk_forward:
    rolling_signals = run_walk_forward(data, contract_prices, config)
    rolling_signals.to_csv(run_dir / "rolling_signals.csv", index=False)
    if not rolling_signals.empty:
        rolling_backtest = BacktestEngine(config, data, contract_prices).run(rolling_signals)
        rolling_backtest.equity.to_csv(run_dir / "rolling_equity.csv")
        display(calculate_metrics(rolling_backtest.returns, config.backtest).to_frame("rolling"))
else:
    print("本轮未启用全品种滚动实验；可在配置中开启。")

## 7. 研究记录

在这里记录假设、对照实验和结论，避免把研究判断写死到库代码中。

- 基准：空仓、固定价差方向或简单阈值策略。
- 敏感性：手续费、窗口长度、网络宽度、随机种子。
- 稳健性：多个未参与调参的时间区间，并报告交易次数与回撤。
- 本轮结论：请结合保存的配置、训练曲线和账户流水填写。

本次运行的配置与依赖版本已保存到结果目录；实现细节见[架构文档](../docs/architecture.md)，行为变化见[迁移说明](../docs/migration.md)。